# 🏆 Notebook 02: Capa Gold - Modelado Estrella (Star Schema)

**Objetivo:** Transformar el dataset normalizado de la Capa Silver en un Modelo Estrella optimizado para Inteligencia de Negocios (Power BI) y consultas analíticas de alto rendimiento.

**Estrategia de Modelado:**
1. **Extracción de Dimensiones (Dim Tables):** Separar los atributos descriptivos (geografía, niveles de gobierno, rubros) y asignarles un ID único (Surrogate Key) para eliminar texto redundante.
2. **Construcción de Hechos (Fact Table):** Crear una tabla central delgada y ultra veloz que contenga únicamente las llaves foráneas (IDs) y las métricas financieras (`monto`).
3. **Exportación Final:** Guardar las tablas estructuradas listas para ser consumidas por un motor analítico (DuckDB).

In [ ]:
### Arquitectura del Modelo Estrella: Orden de Extracción

Para construir nuestro modelo analítico de manera eficiente, extraeremos las dimensiones en un orden lógico, agrupando las columnas del MEF según su naturaleza. Al final, construiremos la Tabla de Hechos.

**1. Dimensión Geografía (dim_geografia)**
* **Objetivo:** Normalizar la ubicación espacial donde se ejecuta el presupuesto.
* **Atributos:** Departamento, Provincia, Distrito.

**2. Dimensión Institucional (dim_institucion)**
* **Objetivo:** Mapear la jerarquía de las entidades del Estado (quién gasta el dinero).
* **Atributos:** Nivel de Gobierno, Sector, Pliego, Unidad Ejecutora.

**3. Dimensión Programática (dim_programatica)**
* **Objetivo:** Clasificar el propósito de la inversión pública (en qué se gasta).
* **Atributos:** Categoría Presupuestal, Programa, Producto/Proyecto, Función, División Funcional.

**4. Dimensión Económica (dim_economica)**
* **Objetivo:** Catalogar la naturaleza contable y financiera del gasto.
* **Atributos:** Tipo de Transacción, Genérica, Subgenérica, Específica.

**5. Tabla de Hechos (fact_presupuesto)**
* **Objetivo:** Ser el núcleo central y ultra veloz del modelo. 
* **Contenido:** Contendrá únicamente las claves subrogadas (IDs numéricos) de todas las dimensiones anteriores, el Año, la Fase financiera y la métrica de negocio (`monto`).

In [2]:
import polars as pl
from pathlib import Path

# 1. Definicion de rutas del proyecto
RUTA_SILVER = Path("../data/02_silver/mef_final_silver.parquet")
RUTA_GOLD_DIR = Path("../data/03_gold")

RUTA_GOLD_DIR.mkdir(parents=True, exist_ok=True)

print("Iniciando la extraccion de la Dimension Geografia...")

# 2. Lectura diferida (Lazy Execution)
lazy_silver = pl.scan_parquet(RUTA_SILVER)

# 3. Seleccion de atributos de la dimension (CORREGIDO CON EL ESQUEMA REAL)
# Incluimos tanto los codigos como los nombres descriptivos
columnas_geo = [
    "departamento_ejecutora", "departamento_ejecutora_nombre",
    "provincia_ejecutora", "provincia_ejecutora_nombre",
    "distrito_ejecutora", "distrito_ejecutora_nombre"
]

# 4. Transformacion y normalizacion
dim_geografia = (
    lazy_silver
    .select(columnas_geo)
    # unique() elimina las millones de repeticiones
    .unique()
    # drop_nulls elimina filas donde el codigo del departamento este vacio
    .drop_nulls(subset=["departamento_ejecutora"])
)

# 5. Generacion de la Llave Subrogada (Surrogate Key)
# Usamos los codigos del MEF concatenados para generar un Hash unico y robusto
dim_geografia = dim_geografia.with_columns(
    (pl.col("departamento_ejecutora") + "_" + 
     pl.col("provincia_ejecutora") + "_" + 
     pl.col("distrito_ejecutora"))
    .hash()
    .alias("sk_geografia_id")
)

# 6. Ordenamiento de la estructura
columnas_ordenadas = ["sk_geografia_id"] + columnas_geo
dim_geografia = dim_geografia.select(columnas_ordenadas)

# 7. Materializacion y guardado
# Al reducir la dimensionalidad, procesamos en RAM sin problemas
df_dim_geografia = dim_geografia.collect()

ruta_salida_geo = RUTA_GOLD_DIR / "dim_geografia.parquet"
df_dim_geografia.write_parquet(ruta_salida_geo, compression="zstd")

print(f"Dimension completada. Registros unicos extraidos: {df_dim_geografia.shape[0]}")
print(f"Archivo optimizado guardado en: {ruta_salida_geo}")

Iniciando la extraccion de la Dimension Geografia...
Dimension completada. Registros unicos extraidos: 1892
Archivo optimizado guardado en: ../data/03_gold/dim_geografia.parquet


In [5]:
#Dimension Institucional
import polars as pl
from pathlib import Path

# 1. Rutas (reutilizamos las variables para mantener consistencia)
RUTA_SILVER = Path("../data/02_silver/mef_final_silver.parquet")
RUTA_GOLD_DIR = Path("../data/03_gold")

print("Iniciando la extraccion de la Dimension Institucional...")

# 2. Lectura diferida
lazy_silver = pl.scan_parquet(RUTA_SILVER)

# 3. Definicion del esquema institucional
# Extraemos la jerarquia completa basandonos en el listado real de columnas del MEF.
# 'sec_ejec' es el codigo unico maestro que identifica a cualquier Unidad Ejecutora en el Peru.
columnas_inst = [
    "nivel_gobierno", "nivel_gobierno_nombre",
    "sector", "sector_nombre",
    "pliego", "pliego_nombre",
    "sec_ejec", "ejecutora", "ejecutora_nombre"
]

# 4. Transformacion y normalizacion
dim_institucion = (
    lazy_silver
    .select(columnas_inst)
    .unique()
    # Eliminamos filas donde el codigo de nivel de gobierno no exista
    .drop_nulls(subset=["nivel_gobierno"])
)

# 5. Generacion de la Llave Subrogada (Surrogate Key)
# Concatenamos los codigos duros (no los nombres) para generar el hash.
dim_institucion = dim_institucion.with_columns(
    (pl.col("nivel_gobierno") + "_" + 
     pl.col("sector") + "_" + 
     pl.col("pliego") + "_" + 
     pl.col("sec_ejec"))
    .hash()
    .alias("sk_institucion_id")
)

# 6. Ordenamiento
columnas_ordenadas = ["sk_institucion_id"] + columnas_inst
dim_institucion = dim_institucion.select(columnas_ordenadas)

# 7. Materializacion y guardado
df_dim_institucion = dim_institucion.collect()

ruta_salida_inst = RUTA_GOLD_DIR / "dim_institucion.parquet"
df_dim_institucion.write_parquet(ruta_salida_inst, compression="zstd")

print(f"Dimension Institucional completada. Registros unicos extraidos: {df_dim_institucion.shape[0]}")
print(f"Archivo optimizado guardado en: {ruta_salida_inst}")

Iniciando la extraccion de la Dimension Institucional...
Dimension Institucional completada. Registros unicos extraidos: 2905
Archivo optimizado guardado en: ../data/03_gold/dim_institucion.parquet


In [1]:
import polars as pl
from pathlib import Path

# 1. Definicion de rutas
RUTA_SILVER = Path("../data/02_silver/mef_final_silver.parquet")
RUTA_GOLD_DIR = Path("../data/03_gold")
ruta_salida_prog = RUTA_GOLD_DIR / "dim_programatica.parquet"

print("Iniciando la extraccion Programatica en MODO STREAMING (Anti-Crash)...")

# 2. Lectura diferida
lazy_silver = pl.scan_parquet(RUTA_SILVER)

# 3. Esquema programatico
columnas_prog = [
    "programa_ppto", "programa_ppto_nombre",
    "tipo_act_proy", "tipo_act_proy_nombre",
    "producto_proyecto", "producto_proyecto_nombre",
    "actividad_accion_obra", "actividad_accion_obra_nombre",
    "funcion", "funcion_nombre",
    "division_funcional", "division_funcional_nombre",
    "grupo_funcional", "grupo_funcional_nombre",
    "meta", "meta_nombre"
]

# 4 y 5. Transformacion y Generacion de Llave
dim_programatica = (
    lazy_silver
    .select(columnas_prog)
    .filter(pl.col("programa_ppto").is_not_null())
    # maintain_order=False es vital aquí: le quita el peso a la RAM de recordar el orden original
    .unique(maintain_order=False) 
    .with_columns(
        pl.concat_str(
            [
                pl.col("programa_ppto"), pl.col("producto_proyecto"), 
                pl.col("actividad_accion_obra"), pl.col("funcion"), 
                pl.col("division_funcional"), pl.col("grupo_funcional"),
                pl.col("meta")
            ], 
            separator="_"
        )
        .hash()
        .alias("sk_programatica_id")
    )
    .select(["sk_programatica_id"] + columnas_prog)
)

# 6. Materializacion PROGRESIVA (Directo a disco duro)
print("Procesando en lotes y escribiendo directo al disco...")
dim_programatica.sink_parquet(ruta_salida_prog, compression="zstd")

# 7. Verificacion ligera (Solo lee los metadatos sin cargar la tabla)
total_registros = pl.scan_parquet(ruta_salida_prog).select(pl.len()).collect().item()

print(f"✅ ¡Completado sin reventar la RAM! Registros unicos extraidos: {total_registros}")
print(f"Archivo optimizado guardado en: {ruta_salida_prog}")

Iniciando la extraccion Programatica en MODO STREAMING (Anti-Crash)...
Procesando en lotes y escribiendo directo al disco...
✅ ¡Completado sin reventar la RAM! Registros unicos extraidos: 345402
Archivo optimizado guardado en: ../data/03_gold/dim_programatica.parquet


In [2]:
import polars as pl
from pathlib import Path

# 1. Definicion de rutas
RUTA_SILVER = Path("../data/02_silver/mef_final_silver.parquet")
RUTA_GOLD_DIR = Path("../data/03_gold")
ruta_salida_eco = RUTA_GOLD_DIR / "dim_economica.parquet"

print("Iniciando la extraccion de la Dimension Economica...")

# 2. Lectura diferida
lazy_silver = pl.scan_parquet(RUTA_SILVER)

# 3. Esquema economico (Clasificador de Gastos)
# Extraemos la jerarquia contable basandonos en el esquema de la capa Silver.
# Nota: 'tipo_transaccion' usualmente no tiene columna '_nombre' asociada.
columnas_eco = [
    "tipo_transaccion", 
    "generica", "generica_nombre",
    "subgenerica", "subgenerica_nombre",
    "subgenerica_det", "subgenerica_det_nombre",
    "especifica", "especifica_nombre",
    "especifica_det", "especifica_det_nombre"
]

# 4 y 5. Transformacion y Generacion de Llave Subrogada
dim_economica = (
    lazy_silver
    .select(columnas_eco)
    .filter(pl.col("tipo_transaccion").is_not_null())
    .unique(maintain_order=False) 
    .with_columns(
        # Usamos concat_str solo con los codigos (no los nombres) para generar un hash robusto
        pl.concat_str(
            [
                pl.col("tipo_transaccion"),
                pl.col("generica"),
                pl.col("subgenerica"),
                pl.col("subgenerica_det"),
                pl.col("especifica"),
                pl.col("especifica_det")
            ], 
            separator="_"
        )
        .hash()
        .alias("sk_economica_id")
    )
    .select(["sk_economica_id"] + columnas_eco)
)

# 6. Materializacion Progresiva (Streaming)
print("Procesando y escribiendo directo al disco...")
dim_economica.sink_parquet(ruta_salida_eco, compression="zstd")

# 7. Verificacion de registros
total_registros_eco = pl.scan_parquet(ruta_salida_eco).select(pl.len()).collect().item()

print(f"Dimension Economica completada. Registros unicos extraidos: {total_registros_eco}")
print(f"Archivo optimizado guardado en: {ruta_salida_eco}")

Iniciando la extraccion de la Dimension Economica...
Procesando y escribiendo directo al disco...
Dimension Economica completada. Registros unicos extraidos: 579
Archivo optimizado guardado en: ../data/03_gold/dim_economica.parquet


In [4]:
import polars as pl
from pathlib import Path

# 1. Definicion de rutas
RUTA_SILVER = Path("../data/02_silver/mef_final_silver.parquet")
RUTA_GOLD_DIR = Path("../data/03_gold")
ruta_salida_fin = RUTA_GOLD_DIR / "dim_financiamiento.parquet"

print("Iniciando la extraccion de la Dimension de Financiamiento...")

# 2. Lectura diferida
lazy_silver = pl.scan_parquet(RUTA_SILVER)

# 3. Esquema de Financiamiento
# Incluimos las fuentes de ingresos y el tipo de recurso segun el esquema del MEF
columnas_fin = [
    "fuente_financiamiento", "fuente_financiamiento_nombre",
    "rubro", "rubro_nombre",
    "tipo_recurso", "tipo_recurso_nombre",
    "categoria_gasto", "categoria_gasto_nombre"
]

# 4 y 5. Transformacion y Generacion de Llave Subrogada
dim_financiamiento = (
    lazy_silver
    .select(columnas_fin)
    .filter(pl.col("fuente_financiamiento").is_not_null())
    .unique(maintain_order=False) 
    .with_columns(
        # Concatenamos los codigos duros para el identificador unico
        pl.concat_str(
            [
                pl.col("fuente_financiamiento"),
                pl.col("rubro"),
                pl.col("tipo_recurso"),
                pl.col("categoria_gasto")
            ], 
            separator="_"
        )
        .hash()
        .alias("sk_financiamiento_id")
    )
    .select(["sk_financiamiento_id"] + columnas_fin)
)

# 6. Materializacion Progresiva (Streaming)
print("Procesando en lotes y escribiendo directo al disco...")
dim_financiamiento.sink_parquet(ruta_salida_fin, compression="zstd")

# 7. Verificacion de registros
total_registros_fin = pl.scan_parquet(ruta_salida_fin).select(pl.len()).collect().item()

print(f"Dimension Financiamiento completada. Registros unicos extraidos: {total_registros_fin}")
print(f"Archivo optimizado guardado en: {ruta_salida_fin}")

Iniciando la extraccion de la Dimension de Financiamiento...
Procesando en lotes y escribiendo directo al disco...
Dimension Financiamiento completada. Registros unicos extraidos: 316
Archivo optimizado guardado en: ../data/03_gold/dim_financiamiento.parquet
